In [ ]:
# Cell 1: Install & Imports
!pip install -q transformers datasets accelerate scikit-learn

import os
import json
import shutil
import torch
import pandas as pd
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Cell 2: Dataset Definition
class TicketDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.reset_index(drop=True) if hasattr(texts, 'reset_index') else texts
        self.labels = labels.reset_index(drop=True) if hasattr(labels, 'reset_index') else labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# Cell 3: Load Data & Preprocessing
# Note: Upload your local 'data/processed/tickets.csv' to the Kaggle notebook input
df = pd.read_csv('/kaggle/input/support-tickets/tickets.csv')

labels = sorted(df['queue'].unique().tolist())
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}
df['label'] = df['queue'].map(label2id)

os.makedirs("models/distilbert", exist_ok=True)
with open("models/distilbert/queue_label_mapping.json", "w") as f:
    json.dump(label2id, f, indent=4)

X_train, X_temp, y_train, y_temp = train_test_split(df['text'], df['label'], test_size=0.3, random_state=42, stratify=df['label'])
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_loader = DataLoader(TicketDataset(X_train, y_train, tokenizer), batch_size=16, shuffle=True)
val_loader = DataLoader(TicketDataset(X_val, y_val, tokenizer), batch_size=16)
test_loader = DataLoader(TicketDataset(X_test, y_test, tokenizer), batch_size=16)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

In [ ]:
# Cell 4: Training & Validation Loop (Matching PDF Section 10)
epochs = 5
best_val_f1 = 0.0

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    print("-" * 30)
    
    model.train()
    train_loss, train_preds, train_labels = 0, [], []
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        batch_labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=batch_labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        train_preds.extend(preds)
        train_labels.extend(batch_labels.cpu().numpy())

    t_acc = accuracy_score(train_labels, train_preds)
    t_f1 = f1_score(train_labels, train_preds, average='macro')
    print(f"Train Loss: {train_loss/len(train_loader):.3f} | Train Acc: {t_acc*100:.2f}% | Train Macro F1: {t_f1*100:.2f}%")

    model.eval()
    val_loss, val_preds, val_labels = 0, [], []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            batch_labels = batch['labels'].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=batch_labels)
            val_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_labels.extend(batch_labels.cpu().numpy())

    v_acc = accuracy_score(val_labels, val_preds)
    v_f1 = f1_score(val_labels, val_preds, average='macro')
    print(f"Validation Loss: {val_loss/len(val_loader):.3f} | Validation Acc: {v_acc*100:.2f}% | Validation Macro F1: {v_f1*100:.2f}%")

In [ ]:
# Cell 5: Final Evaluation & Archive Export (Matching PDF Section 11-12)
print("\nFINAL TEST EVALUATION")
model.eval()
test_preds, test_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        batch_labels = batch['labels'].to(device)
        outputs = model(input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_labels.extend(batch_labels.cpu().numpy())

test_acc = accuracy_score(test_labels, test_preds)
test_macro = f1_score(test_labels, test_preds, average='macro')
test_weighted = f1_score(test_labels, test_preds, average='weighted')

print(f"Test Accuracy: {test_acc*100:.2f}%")
print(f"Test Macro F1: {test_macro*100:.2f}%")
print(f"Test Weighted F1: {test_weighted*100:.2f}%")

model.save_pretrained("models/distilbert", safe_serialization=True)
tokenizer.save_pretrained("models/distilbert")

# Create zip archive for download as specified in PDF Section 12
shutil.make_archive("/kaggle/working/distilbert_artifacts", "zip", "models/distilbert")
print("Archive created at /kaggle/working/distilbert_artifacts.zip")